# PayWave Customer Inactivity Prediction — Guided Analysis

Build a leakage-safe, temporally validated decision system for a weekly retention team with capacity for 5,000 customer contacts.

## Learning objectives

This notebook demonstrates target design, data-quality validation, leakage prevention, chronological model validation, baseline comparison, calibration review, capacity-aware evaluation, segment diagnostics, business-value estimation, and a pilot recommendation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

MODULE_ROOT = Path.cwd()
if MODULE_ROOT.name == 'notebooks':
    MODULE_ROOT = MODULE_ROOT.parent
sys.path.insert(0, str(MODULE_ROOT))

from src.generate_synthetic_data import generate_raw_data
from src.train_evaluate_models import (
    TARGET, OUTREACH_CAPACITY, LEAKAGE_COLUMNS, build_models, clean_data,
    estimate_business_value, fit_and_score, quality_report, segment_metrics,
    temporal_split,
)

pd.set_option('display.max_columns', None)

## 1. Frame the prediction decision

At each scoring date, estimate whether an eligible active customer will record zero qualifying transactions in the next 30 days. The score supports a ranked outreach list; it does not determine whether a particular intervention will cause a customer to remain active.

In [ ]:
raw = generate_raw_data()
print(f'Raw rows: {len(raw):,}')
display(raw.head())

## 2. Validate data quality and remove leakage

The raw data intentionally contain defects and three future-information fields. They must be identified before model training.

In [ ]:
display(pd.Series(quality_report(raw), name='count').to_frame())
print('Leakage fields:', [c for c in LEAKAGE_COLUMNS if c in raw.columns])
display((raw.isna().mean().mul(100).sort_values(ascending=False).head(8)).to_frame('missing_pct'))

In [ ]:
df = clean_data(raw)
print(f'Analytical rows: {len(df):,}')
assert not df.duplicated(['customer_id', 'scoring_date']).any()
assert set(df[TARGET].unique()).issubset({0, 1})

## 3. Split chronologically

Train on the oldest cohorts, select the model on later Validation cohorts, and reserve August as the final Test cohort. This mirrors the deployment question more closely than a random split.

In [ ]:
train, validation, test = temporal_split(df)
split_summary = pd.DataFrame({
    name: {'rows': len(part), 'start': part.scoring_date.min(), 'end': part.scoring_date.max(), 'prevalence': part[TARGET].mean()}
    for name, part in [('Train', train), ('Validation', validation), ('Test', test)]
}).T
display(split_summary)

In [ ]:
cohort_prevalence = df.groupby('scoring_date')[TARGET].mean()
ax = cohort_prevalence.mul(100).plot(marker='o', figsize=(9, 4), title='Inactivity prevalence by scoring cohort')
ax.set_ylabel('Inactive next 30 days (%)')
ax.set_xlabel('Scoring date')
plt.show()

## 4. Compare baselines and candidate models

All preprocessing is contained in pipelines fit on Train only. Validation covers two scoring cycles, so its capacity is 10,000 contacts in total. Test represents one 5,000-contact cycle.

In [ ]:
results = fit_and_score(build_models(), train, validation, test)
validation_table = results['validation_table'].copy()
display(validation_table.round(4))
print('Selected on Validation PR AUC:', results['selected_name'])

In [ ]:
ax = validation_table['pr_auc'].sort_values().plot.barh(figsize=(8, 4), color='#2F6B8A')
ax.set_title('Validation PR AUC by candidate')
ax.set_xlabel('PR AUC')
plt.show()

## 5. Evaluate the frozen model on Test

The held-out cohort is evaluated once. Capacity metrics answer the operational question directly.

In [ ]:
test_metrics = pd.Series(results['test_metrics'].__dict__, name='selected_model')
rule_metrics = pd.Series(results['rule_test_metrics'].__dict__, name='business_rule')
display(pd.concat([test_metrics, rule_metrics], axis=1).round(4))
incremental_captures = results['test_metrics'].captured_positives - results['rule_test_metrics'].captured_positives
print(f'Incremental inactive customers captured at equal capacity: {incremental_captures:,}')

## 6. Review probability calibration

Brier loss provides one summary, but decile-level observed versus predicted rates make calibration easier to diagnose. The business-rule score is not a probability and is therefore excluded.

In [ ]:
calibration = test[[TARGET]].copy()
calibration['score'] = results['test_scores']
calibration['decile'] = pd.qcut(calibration['score'], 10, duplicates='drop')
calibration_table = calibration.groupby('decile', observed=True).agg(
    customers=(TARGET, 'size'), predicted_rate=('score', 'mean'), observed_rate=(TARGET, 'mean')
)
display(calibration_table)

## 7. Evaluate important segments

Interpret PR AUC alongside prevalence and sample size. Differences in prevalence change the PR baseline.

In [ ]:
for segment in ['country', 'device_type', 'customer_value_tier']:
    print(f'\n{segment}')
    display(segment_metrics(test, results['test_scores'], segment).round(4))

## 8. Estimate illustrative business value

The following is a transparent sensitivity scenario—not realized impact. Retention intervention effectiveness requires separate causal validation.

In [ ]:
base_value = estimate_business_value(results['test_metrics'])
display(pd.Series(base_value, name='base_case').to_frame())

sensitivity = []
for success_rate in [0.10, 0.18, 0.25]:
    row = estimate_business_value(results['test_metrics'], intervention_success_rate=success_rate)
    sensitivity.append({'intervention_success_rate': success_rate, **row})
display(pd.DataFrame(sensitivity).set_index('intervention_success_rate').round(2))

## 9. Recommendation

**Proceed to a controlled operational pilot.** Gradient Boosting improves capacity-constrained targeting over the current recency rule and performs consistently across major segments. Before wider deployment, validate intervention lift, confirm production feature availability, review calibration, monitor customer and segment outcomes, and define drift, retraining, and rollback thresholds.

## Analyst QA checklist

- [ ] Target, horizon, population, and action are reproducible.
- [ ] Every feature is available before scoring.
- [ ] Preprocessing is fit on Train only.
- [ ] Model selection does not use Test.
- [ ] Rankings and probabilities are interpreted separately.
- [ ] Capacity metrics match the operating cycle.
- [ ] Segment results include prevalence and sample size.
- [ ] Business value uses visible assumptions.
- [ ] Predictive performance is not described as causal intervention impact.